In [20]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display
from collections import namedtuple

pd.options.display.max_columns = None

In [3]:
fangraphs_board = pd.read_csv('pitching_1947_2001.csv')
fangraphs_board.rename(columns={'season': 'yearID', 'team': 'teamID'}, inplace = True)
fangraphs_board.drop(['HR/FB', 'xFIP', 'xFIP-', 'playerid', 'K/9', 'BB/9', 'K/9+', 'BB/9+'], axis=1, inplace=True)

## Here We Go!

The first thing that I want to do, is to group everybody into teams. In order to do this analysis, I'm going to be looking at Johnson and Schilling's numbers together as a single aggregate. I think that would be the best, becuase I'm going to compare him to other top 2 pitchers on teams so we're really comparing "groups" of players rather than individual players.

### The Plan

What I plan on doing is first, create the aggregate values for each column, for the top two pitchers in a team each year. After I do that I want to rank each pair of staff aces for each statistic. From there, create an average score for how they rank in each statistic to find the most "dominant" pairing of pitchers. Then, look across each year and see who has the highest "dominance" score.

In [4]:
fangraphs_board = stat.order_by(fangraphs_board, ['yearID', 'teamID'], True)
fangraphs_board = fangraphs_board[fangraphs_board.teamID != '- - -']

In [10]:
year2001 = fangraphs_board[fangraphs_board.yearID == 2001].reset_index(drop=True)
year2001.head()

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K%,BB%,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
0,2001,Jarrod Washburn,Angels,11,10,30,30,193.1,1.29,15.5%,6.6%,1.16,0.285,3.77,4.37,85,98,117.0,93,94,80,2.5
1,2001,Ramon Ortiz,Angels,13,11,32,32,208.2,1.43,14.7%,8.3%,1.08,0.296,4.36,4.58,98,103,89.0,103,89,100,2.3
2,2001,Ismael Valdez,Angels,9,13,27,27,163.2,1.39,14.3%,7.2%,1.10,0.301,4.45,4.48,100,101,100.0,100,87,86,2.0
3,2001,Pat Rapp,Angels,5,12,28,28,165.0,1.41,11.0%,9.7%,1.04,0.268,4.80,4.89,108,110,57.0,102,67,117,1.3
4,2001,Scott Schoeneweis,Angels,10,11,32,32,205.1,1.48,11.4%,8.5%,0.92,0.297,5.08,4.70,114,106,68.0,106,69,102,2.0


In [11]:
teams_w_mult = {}
for team in year2001.teamID.unique():
    counter = 0
    teams_df = year2001[year2001.teamID == team]
    for name in teams_df.name.unique():
        counter += 1
    teams_w_mult[team] = counter
print(teams_w_mult)

{'Angels': 5, 'Astros': 2, 'Athletics': 4, 'Blue Jays': 2, 'Braves': 3, 'Brewers': 2, 'Cardinals': 3, 'Cubs': 4, 'Devil Rays': 1, 'Diamondbacks': 2, 'Dodgers': 1, 'Expos': 2, 'Giants': 3, 'Indians': 2, 'Mariners': 3, 'Marlins': 4, 'Mets': 4, 'Orioles': 2, 'Padres': 2, 'Phillies': 2, 'Pirates': 2, 'Rangers': 2, 'Red Sox': 1, 'Reds': 2, 'Rockies': 2, 'Royals': 2, 'Tigers': 2, 'Twins': 3, 'White Sox': 1, 'Yankees': 3}


In [12]:
angels_2001 = year2001[year2001.teamID == 'Angels']
angels_2001 = angels_2001.head(2)
angels_2001

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K%,BB%,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
0,2001,Jarrod Washburn,Angels,11,10,30,30,193.1,1.29,15.5%,6.6%,1.16,0.285,3.77,4.37,85,98,117.0,93,94,80,2.5
1,2001,Ramon Ortiz,Angels,13,11,32,32,208.2,1.43,14.7%,8.3%,1.08,0.296,4.36,4.58,98,103,89.0,103,89,100,2.3


In [13]:
dont_do = ['K%', 'BB%', 'name', 'teamID']
averages_dict = {}
for col in angels_2001.columns:
    if col not in dont_do:
        averages_dict[col] = angels_2001[col].mean()
print(averages_dict)

{'yearID': 2001.0, 'W': 12.0, 'L': 10.5, 'G': 31.0, 'GS': 31.0, 'IP': 200.64999999999998, 'WHIP': 1.3599999999999999, 'HR/9': 1.12, 'BABIP': 0.2905, 'ERA': 4.065, 'FIP': 4.475, 'ERA-': 91.5, 'FIP-': 100.5, 'K/BB+': 103.0, 'WHIP+': 98.0, 'K%+': 91.5, 'BB%+': 90.0, 'WAR': 2.4}


In [15]:
diamondbacks_2001 = fangraphs_board[(fangraphs_board.teamID == 'Diamondbacks') & (fangraphs_board.yearID == 2001)]
diamondbacks_avgs = {}
for col in diamondbacks_2001.columns:
    if col not in dont_do:
        diamondbacks_avgs[col] = diamondbacks_2001[col].mean()
print(diamondbacks_avgs)

{'yearID': 2001.0, 'W': 21.0, 'L': 6.0, 'G': 34.5, 'GS': 34.5, 'IP': 249.2, 'WHIP': 1.0550000000000002, 'HR/9': 1.0, 'BABIP': 0.312, 'ERA': 2.77, 'FIP': 2.665, 'ERA-': 61.5, 'FIP-': 59.0, 'K/BB+': 300.0, 'WHIP+': 77.0, 'K%+': 181.0, 'BB%+': 64.0, 'WAR': 8.45}


In [16]:
fangraphs_board.head()

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K%,BB%,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
4,1947,Dick Fowler,Athletics,12,11,36,31,227.1,1.30,8.0%,9.0%,0.48,0.258,2.81,3.68,76,97,90.0,93,82,90,2.9
5,1947,Phil Marchildon,Athletics,19,9,35,35,276.2,1.33,10.9%,12.0%,0.49,0.242,3.22,3.88,87,103,93.0,95,112,121,2.9
6,1947,Bill McCahan,Athletics,10,5,29,19,165.1,1.34,6.7%,8.8%,0.38,0.261,3.32,3.60,90,95,78.0,96,69,88,2.1
7,1947,Joe Coleman,Athletics,6,12,32,21,160.1,1.45,9.4%,9.0%,0.95,0.283,4.32,4.22,117,112,107.0,104,97,90,0.9
8,1947,Warren Spahn,Braves,21,10,40,35,289.2,1.14,10.5%,7.2%,0.47,0.242,2.33,3.20,58,81,145.0,79,110,76,6.0


In [19]:
columns = []
for col in fangraphs_board.columns:
    if col not in dont_do:
        columns.append(col)

In [21]:
years = fangraphs_board.yearID.unique()
teams = fangraphs_board.teamID.unique()
team_dict = namedtuple('team_avgs', 'col_name dict')
def make_avgs_dict(year, team):
    team_dict = {}
    team_df = fangraphs_board[(fangraphs_board.yearID == year) & (fangraphs_board.teamID == team)]
    team_df = stat.order_by(team_df, 'WAR', False)
    team_df = team_df.head(2)
    for col in team_df.columns:
        if col not in dont_do:
            team_dict[col] = team_df[col].mean()
dicts = {}
for year in years:
    for team in teams:
        t = team_dict(col_name=f'{team}_{year}', dict=make_avgs_dict(year, team))
        dicts[t.col_name] = [t.dict]
